# Silver Layer 

In [0]:
import logging

# Configure logger
logger = logging.getLogger("bronze_extraction")
logger.setLevel(logging.INFO)

# Prevent duplicate logs if cell is rerun
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)


### Load Each File

In [0]:
# Define the root path of the Bronze container in ADLS Gen2
base_path = "abfss://bronze@chdatalake.dfs.core.windows.net"

# List of folders/datasets to be read from the Bronze layer
folders = [
    "ingredients",
    "inventory",
    "items",
    "orders",
    "recipe"
]

# Create an empty dictionary to store DataFrames dynamically
dfs = {}

# Log the start of the extraction process
logger.info("Starting Bronze Data Extraction...")

# Loop through each folder in the Bronze container
for folder in folders:
    try:
        # Log the current folder being processed
        logger.info(f"Reading folder: {folder}")

        # Read CSV files from the current folder into a Spark DataFrame
        df = (
            spark.read
                 .format("csv")
                 .option("header", "true")       # Use first row as column headers
                 .option("inferSchema", "true") # Automatically infer data types
                 .csv(f"{base_path}/{folder}/") # Read files from the folder path
        )

        # Store the DataFrame in the dictionary using the folder name as the key
        dfs[folder] = df

        # Capture metadata for logging purposes
        row_count = df.count()
        column_count = len(df.columns)

        # Log successful extraction details
        logger.info(
            f"Successfully loaded '{folder}' "
            f"| Rows: {row_count:,} "
            f"| Columns: {column_count}"
        )

    except Exception as e:
        # Log any errors that occur during extraction
        logger.error(
            f"Failed to load folder '{folder}'. Error: {str(e)}"
        )

# Log the completion of the Bronze extraction process
logger.info("Bronze Data Extraction Completed.")

2026-06-21 01:14:01,855 | INFO | Starting Bronze Data Extraction...
2026-06-21 01:14:01,856 | INFO | Reading folder: ingredients
2026-06-21 01:14:03,014 | INFO | Successfully loaded 'ingredients' | Rows: 18 | Columns: 5
2026-06-21 01:14:03,016 | INFO | Reading folder: inventory
2026-06-21 01:14:03,901 | INFO | Successfully loaded 'inventory' | Rows: 18 | Columns: 3
2026-06-21 01:14:03,903 | INFO | Reading folder: items
2026-06-21 01:14:04,763 | INFO | Successfully loaded 'items' | Rows: 37 | Columns: 6
2026-06-21 01:14:04,765 | INFO | Reading folder: orders
2026-06-21 01:14:05,602 | INFO | Successfully loaded 'orders' | Rows: 10,000 | Columns: 7
2026-06-21 01:14:05,604 | INFO | Reading folder: recipe
2026-06-21 01:14:06,727 | INFO | Successfully loaded 'recipe' | Rows: 61 | Columns: 4
2026-06-21 01:14:06,729 | INFO | Bronze Data Extraction Completed.


## Transformations

### Ingredients

In [0]:
dfs["ingredients"].display()

ing_id,ing_name,ing_weight,ing_meas,ing_price
ING001,ESPRESSO BEANS,1000,Gram,$12.00
ING002,Whole Milk,1000,ML,1.2 USD
ING003,"""""""CHEDDAR""""""",500,Gram,7.45 USD
ING004,""""""" mozzarella """"""",500,GRAMS,$5.00
ING005,whipped cream,300,mL,1.35 USD
ING006,"""""""Vanilla syrup""""""",1000,ml,$14.52
ING007,barista chocolate syrup,""""""" 1000 """"""",mL,$8.49
ING008,Barista white chocolate syrup,"""""""1000"""" """,milliliters,8.49 USD
ING009,Barista caramel sauce,1000,"""""""ml""""""",8.49 USD
ING010,Sugar,1000,grms,$1.50


In [0]:
## Import libraries
from pyspark.sql.functions import *
from functools import reduce
from pyspark.sql.window import Window

In [0]:
# special characters and trim function

def specialchar_trim(df, columns):
    for column in columns:
        df = df.withColumn(
            column,
            initcap(
                trim(
                    regexp_replace(
                        regexp_replace(
                            col(column),
                            "[^a-zA-Z0-9., -]",
                            " "
                        ),
                        r"\s+",
                        " "
                    )
                )
            )
        )
    return df

In [0]:
# remove special characters and trim the values of each column
df_ingredients = specialchar_trim(
    dfs["ingredients"],
    ["ing_name", "ing_weight", "ing_meas", "ing_price"]
)

display(df_ingredients)

ing_id,ing_name,ing_weight,ing_meas,ing_price
ING001,Espresso Beans,1000,Gram,12.00
ING002,Whole Milk,1000,Ml,1.2 Usd
ING003,Cheddar,500,Gram,7.45 Usd
ING004,Mozzarella,500,Grams,5.00
ING005,Whipped Cream,300,Ml,1.35 Usd
ING006,Vanilla Syrup,1000,Ml,14.52
ING007,Barista Chocolate Syrup,1000,Ml,8.49
ING008,Barista White Chocolate Syrup,1000,Milliliters,8.49 Usd
ING009,Barista Caramel Sauce,1000,Ml,8.49 Usd
ING010,Sugar,1000,Grms,1.50


In [0]:
df_ingredients.display(5)

ing_id,ing_name,ing_weight,ing_meas,ing_price
ING001,Espresso Beans,1000,Gram,12.00
ING002,Whole Milk,1000,Ml,1.2 Usd
ING003,Cheddar,500,Gram,7.45 Usd
ING004,Mozzarella,500,Grams,5.00
ING005,Whipped Cream,300,Ml,1.35 Usd
ING006,Vanilla Syrup,1000,Ml,14.52
ING007,Barista Chocolate Syrup,1000,Ml,8.49
ING008,Barista White Chocolate Syrup,1000,Milliliters,8.49 Usd
ING009,Barista Caramel Sauce,1000,Ml,8.49 Usd
ING010,Sugar,1000,Grms,1.50


In [0]:
# remove Usd after price amount
df_ingredients = df_ingredients.withColumn("ing_price", regexp_replace(col("ing_price"), "Usd", " "))

In [0]:
# standardize unit measurement
df_ingredients = df_ingredients.withColumn("ing_meas", 
when(col("ing_meas") == "Gram", "G")
.when(col("ing_meas") == "Grms", "G")
.when(col("ing_meas") == "Grams", "G")
.when(col("ing_meas") == "Milliliters", "M")
.when(col("ing_meas") == "MI", "M")
.otherwise(col("ing_meas"))
)

In [0]:
df_ingredients.display()

ing_id,ing_name,ing_weight,ing_meas,ing_price
ING001,Espresso Beans,1000,G,12.00
ING002,Whole Milk,1000,Ml,1.2
ING003,Cheddar,500,G,7.45
ING004,Mozzarella,500,G,5.00
ING005,Whipped Cream,300,Ml,1.35
ING006,Vanilla Syrup,1000,Ml,14.52
ING007,Barista Chocolate Syrup,1000,Ml,8.49
ING008,Barista White Chocolate Syrup,1000,M,8.49
ING009,Barista Caramel Sauce,1000,Ml,8.49
ING010,Sugar,1000,G,1.50


### Inventory

In [0]:
dfs["inventory"].display()

inv_id,ing_id,quantity
inv001,ING001,4
inv002,ING002,55
inv003,ING003,1
inv004,ING004,4
inv005,ING005,7
inv006,ING006,3
inv007,ING007,3
inv008,ING008,4
inv009,ING009,1
inv010,ING010,4


In [0]:
df_inventory = dfs["inventory"]


### Items

In [0]:
display(dfs["items"].limit(10))

item_id,sku,item_name,item_cat,item_size,item_price
It001,HDR-CAP-MD,Cappuccino,Hot Drinks,Medium,3.45
It002,HDR-CAP-LG,Cappuccino,hot drinks,large,$3.75
It003,HDR-LAT-MD,Latte,HOT DRINKS,MED,3.45
It004,HDR-LAT-LG,Latte,Hot beverage,Lrg.,3.75 USD
It005,HDR-FLT,Flat White,Hot_Drinks,Not Applicable,$3.15
It006,HDR-CRM-MD,Caramel Macchiato,Hot Drinks,Medium,4.2
It007,HDR-CRM-LG,Caramel Macchiato,hot drinks,large,$4.60
It008,HDR-ESP,Espresso,HOT DRINKS,null,2.15
It009,HDR-MOC-MD,Mocha,Hot beverage,Med.,4.00 USD
It010,HDR-MOC-LG,Mocha,Hot_Drinks,L,$4.60


In [0]:
# remove special characters and trim the values of each column
df_items = specialchar_trim(dfs["items"], ["item_id", "sku", "item_name", "item_cat", "item_size", "item_price"])

In [0]:
display(df_items.limit(10))

item_id,sku,item_name,item_cat,item_size,item_price
It001,Hdr-cap-md,Cappuccino,Hot Drinks,Medium,3.45
It002,Hdr-cap-lg,Cappuccino,Hot Drinks,Large,3.75
It003,Hdr-lat-md,Latte,Hot Drinks,Med,3.45
It004,Hdr-lat-lg,Latte,Hot Beverage,Lrg.,3.75 Usd
It005,Hdr-flt,Flat White,Hot Drinks,Not Applicable,3.15
It006,Hdr-crm-md,Caramel Macchiato,Hot Drinks,Medium,4.2
It007,Hdr-crm-lg,Caramel Macchiato,Hot Drinks,Large,4.60
It008,Hdr-esp,Espresso,Hot Drinks,null,2.15
It009,Hdr-moc-md,Mocha,Hot Beverage,Med.,4.00 Usd
It010,Hdr-moc-lg,Mocha,Hot Drinks,L,4.60


In [0]:
# remove USD from price amount
df_items = df_items.withColumn("item_price", regexp_replace(col("item_price"), "Usd", ""))

In [0]:
display(df_items.limit(10))

item_id,sku,item_name,item_cat,item_size,item_price
It001,Hdr-cap-md,Cappuccino,Hot Drinks,Medium,3.45
It002,Hdr-cap-lg,Cappuccino,Hot Drinks,Large,3.75
It003,Hdr-lat-md,Latte,Hot Drinks,Med,3.45
It004,Hdr-lat-lg,Latte,Hot Beverage,Lrg.,3.75
It005,Hdr-flt,Flat White,Hot Drinks,Not Applicable,3.15
It006,Hdr-crm-md,Caramel Macchiato,Hot Drinks,Medium,4.2
It007,Hdr-crm-lg,Caramel Macchiato,Hot Drinks,Large,4.60
It008,Hdr-esp,Espresso,Hot Drinks,null,2.15
It009,Hdr-moc-md,Mocha,Hot Beverage,Med.,4.00
It010,Hdr-moc-lg,Mocha,Hot Drinks,L,4.60


In [0]:
# group all sandwhich values into one category
df_items = df_items.withColumn(
    "item_cat",
    when(
        col("item_name").isin(
            "Sandwich Ham Cheese",
            "Turkey And Cheese Sandwich",
            "Egg Sausage And Cheese Crossaint",
            "Sandwich Salami Mozzarella"
        ),
        "Sandwiches"
    ).otherwise(col("item_cat"))
)

In [0]:
display(df_items.limit(10))

item_id,sku,item_name,item_cat,item_size,item_price
It001,Hdr-cap-md,Cappuccino,Hot Drinks,Medium,3.45
It002,Hdr-cap-lg,Cappuccino,Hot Drinks,Large,3.75
It003,Hdr-lat-md,Latte,Hot Drinks,Med,3.45
It004,Hdr-lat-lg,Latte,Hot Beverage,Lrg.,3.75
It005,Hdr-flt,Flat White,Hot Drinks,Not Applicable,3.15
It006,Hdr-crm-md,Caramel Macchiato,Hot Drinks,Medium,4.2
It007,Hdr-crm-lg,Caramel Macchiato,Hot Drinks,Large,4.60
It008,Hdr-esp,Espresso,Hot Drinks,null,2.15
It009,Hdr-moc-md,Mocha,Hot Beverage,Med.,4.00
It010,Hdr-moc-lg,Mocha,Hot Drinks,L,4.60


In [0]:
# update item size and standardize the size for each item
df_items = df_items.withColumn(
    "item_size",
    when(col("item_size").startswith("M"), "M")
    .when(col("item_name").startswith("Espresso"), "1 shot")
    .when(col("item_name").startswith("Avacado"), "Cup")
    .when(col("item_size").startswith("L"), "Lg")
    .when(col("item_size").startswith("Each"), "EA")
    .when(col("item_size").startswith("Not Applicable"), "S")
    .when(col("item_size").startswith("Bottle"), "16 oz bottle")
    .when(col("item_size").isNull(), "EA")
    .when(col("item_size").isin("null","Na", "NA", "N A"), "EA")
    .when(col("item_name").startswith("Pumpkin"), "Bag")
    .otherwise(col("item_size"))
)
  
    

In [0]:
display(df_items.limit(10))

item_id,sku,item_name,item_cat,item_size,item_price
It001,Hdr-cap-md,Cappuccino,Hot Drinks,M,3.45
It002,Hdr-cap-lg,Cappuccino,Hot Drinks,Lg,3.75
It003,Hdr-lat-md,Latte,Hot Drinks,M,3.45
It004,Hdr-lat-lg,Latte,Hot Beverage,Lg,3.75
It005,Hdr-flt,Flat White,Hot Drinks,S,3.15
It006,Hdr-crm-md,Caramel Macchiato,Hot Drinks,M,4.2
It007,Hdr-crm-lg,Caramel Macchiato,Hot Drinks,Lg,4.60
It008,Hdr-esp,Espresso,Hot Drinks,1 shot,2.15
It009,Hdr-moc-md,Mocha,Hot Beverage,M,4.00
It010,Hdr-moc-lg,Mocha,Hot Drinks,Lg,4.60


In [0]:
# update data type for item_price
df_items = df_items.withColumn("item_price", col("item_price").cast("double"))

### Orders

In [0]:
display(dfs["orders"].limit(10))

row_id,order_id,created_at,item_id,quantity,cust_name,in_or_out
1,ORD001,2/12/2024 7:04,It008,1,Avery,out
2,ORD002,2/12/2024 7:09,It014,1,Jordan,in
3,ORD003,2/12/2024 7:14,It008,1,Riley,out
4,ORD004,2/12/2024 7:18,It019,1,Casey,out
5,ORD005,2/12/2024 7:23,It024,1,Skyler,out
6,ORD006,2/12/2024 7:28,It001,1,Morgan,out
7,ORD006,2/12/2024 7:28,It016,1,Morgan,to go
8,ORD007,2/12/2024 7:33,It005,1,Riley,out
9,ORD007,2/12/2024 7:33,It020,1,Riley,null
10,ORD008,2/12/2024 7:39,it006,1,Cameron,OUT


In [0]:
# remove white space and remove special characters from each column
df_orders = specialchar_trim(dfs["orders"], ["row_id", "order_id", "item_id", "quantity", "cust_name", "in_or_out"])
display(df_orders.limit(10))

row_id,order_id,created_at,item_id,quantity,cust_name,in_or_out
1,Ord001,2/12/2024 7:04,It008,1,Avery,Out
2,Ord002,2/12/2024 7:09,It014,1,Jordan,In
3,Ord003,2/12/2024 7:14,It008,1,Riley,Out
4,Ord004,2/12/2024 7:18,It019,1,Casey,Out
5,Ord005,2/12/2024 7:23,It024,1,Skyler,Out
6,Ord006,2/12/2024 7:28,It001,1,Morgan,Out
7,Ord006,2/12/2024 7:28,It016,1,Morgan,To Go
8,Ord007,2/12/2024 7:33,It005,1,Riley,Out
9,Ord007,2/12/2024 7:33,It020,1,Riley,null
10,Ord008,2/12/2024 7:39,It006,1,Cameron,Out


In [0]:
## get schema for orders table
df_orders.printSchema()

root
 |-- row_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- cust_name: string (nullable = true)
 |-- in_or_out: string (nullable = true)



In [0]:
## Find number of rows in orders dataframe

print(df_orders.count())

10000


In [0]:
## create a function to find any null or empty values in a row
def checknulls(df,columns):
    return df.select([
         sum(
            when(col(c).isNull() | (col(c) == ""), 1).otherwise(0))
                .alias(c)
                for c in columns
        ]).show()

checknulls(df_orders, ["row_id", "order_id", "created_at", "item_id", "quantity", "cust_name", "in_or_out"])



+------+--------+----------+-------+--------+---------+---------+
|row_id|order_id|created_at|item_id|quantity|cust_name|in_or_out|
+------+--------+----------+-------+--------+---------+---------+
|     0|       0|        98|     32|     680|      467|      784|
+------+--------+----------+-------+--------+---------+---------+



In [0]:
## create table with incomplete rows with missing data in columns
incomplete_orders = df_orders.filter(
    (col("created_at").isNull()) | (col("created_at") == "") | 
    (col("item_id").isNull()) | (col("created_at") == "") |
    (col("quantity").isNull()) | (col("created_at") == "")

)

display(incomplete_orders.limit(10))


## create separate table with only complete orders
required_cols = ["created_at", "item_id", "quantity"]

condition = reduce(
    lambda x, y: x & y,
    [
        col(c).isNotNull() & (trim(col(c).cast("string")) != "")
        for c in required_cols
    ]
)

complete_orders = df_orders.filter(condition)

display(complete_orders.limit(10))


row_id,order_id,created_at,item_id,quantity,cust_name,in_or_out
75,Ord056,2/12/2024 12:59,It020,null,Chris,Out
80,Ord061,2/12/2024 13:13,It002,null,Jordan,In
93,Ord067,2/12/2024 13:27,It015,null,Jamie,null
196,Ord155,2/13/2024 16:09,It028,null,Casey,
510,Ord427,2/17/2024 14:48,It006,null,Gina,In
529,Ord000529,11/19/2024,It029,null,Alex,Take-out
535,Ord000535,null,It033,4,Chris,Pickup
537,Ord000537,8/29/2024 13:28,It022,null,Jordan,In
540,Ord000540,null,It007,Two,Robin,In
557,Ord000557,6/1/2024 22:36,It029,null,Riley,null


row_id,order_id,created_at,item_id,quantity,cust_name,in_or_out
1,Ord001,2/12/2024 7:04,It008,1,Avery,Out
2,Ord002,2/12/2024 7:09,It014,1,Jordan,In
3,Ord003,2/12/2024 7:14,It008,1,Riley,Out
4,Ord004,2/12/2024 7:18,It019,1,Casey,Out
5,Ord005,2/12/2024 7:23,It024,1,Skyler,Out
6,Ord006,2/12/2024 7:28,It001,1,Morgan,Out
7,Ord006,2/12/2024 7:28,It016,1,Morgan,To Go
8,Ord007,2/12/2024 7:33,It005,1,Riley,Out
9,Ord007,2/12/2024 7:33,It020,1,Riley,null
10,Ord008,2/12/2024 7:39,It006,1,Cameron,Out


In [0]:
# check if there are any nulls in orders table
complete_orders.filter(col("row_id").isNull()).show()
complete_orders.groupBy("row_id")\
    .count()\
        .filter(col("count")> 1)\
            .show()


+------+--------+----------+-------+--------+---------+---------+
|row_id|order_id|created_at|item_id|quantity|cust_name|in_or_out|
+------+--------+----------+-------+--------+---------+---------+
+------+--------+----------+-------+--------+---------+---------+

+------+-----+
|row_id|count|
+------+-----+
+------+-----+



In [0]:
# check to see if there are any nulls for order_id
complete_orders.filter(
    col("order_id").isNull() |
    (trim(col("order_id")) == "")
).show()

complete_orders.filter(
    ~col("order_id").rlike(r"^Ord\d{3}$")
).show()

+------+--------+----------+-------+--------+---------+---------+
|row_id|order_id|created_at|item_id|quantity|cust_name|in_or_out|
+------+--------+----------+-------+--------+---------+---------+
+------+--------+----------+-------+--------+---------+---------+

+------+------------+------------------+-------+--------+---------+---------+
|row_id|    order_id|        created_at|item_id|quantity|cust_name|in_or_out|
+------+------------+------------------+-------+--------+---------+---------+
|   522|   Ord000522| 20-Apr-2024 03:40|  It030|       1|     NULL|   Pickup|
|   523|   Ord000523|        10/25/2024|  It022|       1|      Pat| Take-out|
|   524|   Ord000524|   1/28/2025 23:18|  It028|       2|      Pat|    To Go|
|   525|   Ord000525|    9/23/2024 0:26|  It023|       1|   Morgan| Take-out|
|   526|   Ord000526|   6/30/2024 23:54|  It011|       1|     Alex|       In|
|   527|   Ord000527|    3/2/2024 19:27|  It027|       3|     Alex|     Togo|
|   528|   Ord000528|        2024

In [0]:
##complete_orders = complete_orders.when(col("order_id").startswith("Order-"), regexp_replace("order_id",  ))

from pyspark.sql.functions import when, col, regexp_replace
complete_orders = complete_orders.withColumn("order_id", when(col("order_id").startswith("Order-"), regexp_replace(col("order_id"), "Order-", "")).otherwise(col("order_id")))

display(complete_orders.limit(10))

row_id,order_id,created_at,item_id,quantity,cust_name,in_or_out
1,Ord001,2/12/2024 7:04,It008,1,Avery,Out
2,Ord002,2/12/2024 7:09,It014,1,Jordan,In
3,Ord003,2/12/2024 7:14,It008,1,Riley,Out
4,Ord004,2/12/2024 7:18,It019,1,Casey,Out
5,Ord005,2/12/2024 7:23,It024,1,Skyler,Out
6,Ord006,2/12/2024 7:28,It001,1,Morgan,Out
7,Ord006,2/12/2024 7:28,It016,1,Morgan,To Go
8,Ord007,2/12/2024 7:33,It005,1,Riley,Out
9,Ord007,2/12/2024 7:33,It020,1,Riley,null
10,Ord008,2/12/2024 7:39,It006,1,Cameron,Out


In [0]:
## check to see if Order- was removed from column
complete_orders.createOrReplaceTempView("complete_orders")

spark.sql("""
          SELECT order_id FROM complete_orders where order_id LIKE "Order-$"
          """).show()

+--------+
|order_id|
+--------+
+--------+



In [0]:
# if order_id do not start with ORD, add ORD
complete_orders = complete_orders.withColumn("order_id", when(~col("order_id").startswith("Ord"), 
                                                              concat(lit("Ord"), col("order_id"))
                                                              ).otherwise(col("order_id"))
                                             )

In [0]:
display(complete_orders.limit(10))

row_id,order_id,created_at,item_id,quantity,cust_name,in_or_out
1,Ord001,2/12/2024 7:04,It008,1,Avery,Out
2,Ord002,2/12/2024 7:09,It014,1,Jordan,In
3,Ord003,2/12/2024 7:14,It008,1,Riley,Out
4,Ord004,2/12/2024 7:18,It019,1,Casey,Out
5,Ord005,2/12/2024 7:23,It024,1,Skyler,Out
6,Ord006,2/12/2024 7:28,It001,1,Morgan,Out
7,Ord006,2/12/2024 7:28,It016,1,Morgan,To Go
8,Ord007,2/12/2024 7:33,It005,1,Riley,Out
9,Ord007,2/12/2024 7:33,It020,1,Riley,null
10,Ord008,2/12/2024 7:39,It006,1,Cameron,Out


In [0]:
# check if any order_id values do not have Ord at the start
complete_orders.createOrReplaceTempView("complete_orders")
spark.sql("""
          SELECT * from complete_orders where order_id NOT like 'Ord%'
          """).show()

+------+--------+----------+-------+--------+---------+---------+
|row_id|order_id|created_at|item_id|quantity|cust_name|in_or_out|
+------+--------+----------+-------+--------+---------+---------+
+------+--------+----------+-------+--------+---------+---------+



In [0]:

# standardize created_at column to 'yyyy-MM-dd'
complete_orders = complete_orders.withColumn(
    "created_at",
    to_date(
        coalesce(
            expr("try_to_timestamp(trim(created_at), 'M/d/yyyy H:mm')"),
            expr("try_to_timestamp(trim(created_at), 'M/d/yyyy')"),
            expr("try_to_timestamp(trim(created_at), 'dd-MMM-yyyy HH:mm')"),
            expr("try_to_timestamp(trim(created_at), 'd-MMM-yyyy HH:mm')"),
            expr("try_to_timestamp(trim(created_at), 'yyyy-MM-dd HH:mm:ss')"),
            expr("try_to_timestamp(trim(created_at), 'yyyy-MM-dd')"),
            expr("try_to_timestamp(trim(created_at), 'MM/dd/yyyy hh:mm a')"),
            expr("try_to_timestamp(trim(created_at), 'MM-dd-yyyy HH:mm:ss')"),
            expr("try_to_timestamp(trim(created_at), 'M-d-yyyy H:mm:ss')"),
            expr("try_to_timestamp(trim(created_at), 'dd/MM/yy HH:mm')"),
            expr("try_to_timestamp(trim(created_at), 'yy/MM/dd HH:mm')"),
            expr("try_to_timestamp(trim(created_at), 'yy-MM-dd HH:mm:ss')"),
            expr("try_to_timestamp(trim(created_at), 'yy-MM-dd')"),
            expr("try_to_timestamp(trim(created_at), 'yyyy/MM/dd HH:mm')"),
            expr("try_to_timestamp(trim(created_at), 'yyyy/MM/dd')")
        )
    )
)

display(complete_orders.limit(10))

row_id,order_id,created_at,item_id,quantity,cust_name,in_or_out
1,Ord001,2024-02-12,It008,1,Avery,Out
2,Ord002,2024-02-12,It014,1,Jordan,In
3,Ord003,2024-02-12,It008,1,Riley,Out
4,Ord004,2024-02-12,It019,1,Casey,Out
5,Ord005,2024-02-12,It024,1,Skyler,Out
6,Ord006,2024-02-12,It001,1,Morgan,Out
7,Ord006,2024-02-12,It016,1,Morgan,To Go
8,Ord007,2024-02-12,It005,1,Riley,Out
9,Ord007,2024-02-12,It020,1,Riley,null
10,Ord008,2024-02-12,It006,1,Cameron,Out


In [0]:
# check are there any nulls in created_at column
complete_orders.filter(
    col("created_at").isNull()).count()

0

In [0]:
complete_orders.select("quantity").distinct().limit(10).display()

quantity
3
03
Two
1
4
2


In [0]:
# update any values that are not in int format
complete_orders = complete_orders.withColumn(
    "quantity", 
    when(col("quantity") == "03", "3")
    .when(col("quantity") == "Two", "2")
    .otherwise(col("quantity"))
    )

display(complete_orders.limit(10))

row_id,order_id,created_at,item_id,quantity,cust_name,in_or_out
1,Ord001,2024-02-12,It008,1,Avery,Out
2,Ord002,2024-02-12,It014,1,Jordan,In
3,Ord003,2024-02-12,It008,1,Riley,Out
4,Ord004,2024-02-12,It019,1,Casey,Out
5,Ord005,2024-02-12,It024,1,Skyler,Out
6,Ord006,2024-02-12,It001,1,Morgan,Out
7,Ord006,2024-02-12,It016,1,Morgan,To Go
8,Ord007,2024-02-12,It005,1,Riley,Out
9,Ord007,2024-02-12,It020,1,Riley,null
10,Ord008,2024-02-12,It006,1,Cameron,Out


In [0]:
complete_orders.select("quantity").distinct().display()

quantity
3
1
4
2


In [0]:
complete_orders.select("cust_name").distinct().limit(10).display()

cust_name
Grace
Steve
Ivy
Quin
Umar
Nora
Brooke
Fred
Hugh
Gina


In [0]:
complete_orders.select("in_or_out").distinct().limit(10).display()

in_or_out
In
Togo
null
Dine In
Pickup
Carry Out
Take-out
Out
Take Out
To Go


In [0]:
    # standardize values for in_or_out 
    complete_orders = complete_orders.withColumn(

    "in_or_out",

    when(col("in_or_out") == "In", "Dine In")
    .when(col("in_or_out") == "Togo", "Carry Out")
    .when(col("in_or_out") == "Pickup", "Carry Out")
    .when(col("in_or_out") == "Take-out", "Carry Out")
    .when(col("in_or_out") == "Out", "Carry Out")
    .when(col("in_or_out") == "Take Out", "Carry Out")
    .when(col("in_or_out") == "To Go", "Carry Out")
    .otherwise(col("in_or_out"))
    )

In [0]:
complete_orders.limit(10).display()

row_id,order_id,created_at,item_id,quantity,cust_name,in_or_out
1,Ord001,2024-02-12,It008,1,Avery,Carry Out
2,Ord002,2024-02-12,It014,1,Jordan,Dine In
3,Ord003,2024-02-12,It008,1,Riley,Carry Out
4,Ord004,2024-02-12,It019,1,Casey,Carry Out
5,Ord005,2024-02-12,It024,1,Skyler,Carry Out
6,Ord006,2024-02-12,It001,1,Morgan,Carry Out
7,Ord006,2024-02-12,It016,1,Morgan,Carry Out
8,Ord007,2024-02-12,It005,1,Riley,Carry Out
9,Ord007,2024-02-12,It020,1,Riley,null
10,Ord008,2024-02-12,It006,1,Cameron,Carry Out


In [0]:
complete_orders.select("quantity").distinct().display()

quantity
3
1
4
2


In [0]:
# change data type of column quantity to int
complete_orders = complete_orders.withColumn("quantity", col("quantity").cast("int"))

In [0]:
display(dfs["recipe"].limit(10))

row_id,recipe_id,ing_id,quantity
1,HDR-CAP-MD,ING001,8
2,HDR-CAP-MD,ING002,130
3,HDR-CAP-LG,ING001,10
4,HDR-CAP-LG,ING002,180
5,HDR-LAT-MD,ING001,8
6,HDR-LAT-MD,ING002,130
7,HDR-LAT-LG,ING001,10
8,HDR-LAT-LG,ING002,180
9,HDR-FLT,ING001,8
10,HDR-FLT,ING002,160


In [0]:
df_recipe = dfs["recipe"]

# Load Data

In [0]:
# load transformed data into silver container in datalake
from pyspark.sql import DataFrame
import time
import builtins

silver_path = "abfss://silver@chdatalake.dfs.core.windows.net"

dataframes = {
    "ingredients": df_ingredients,
    "inventory": df_inventory,
    "items": df_items,
    "orders": complete_orders,
    "recipe": df_recipe
}

logger.info("=" * 60)
logger.info("Starting Silver Layer Load...")

for folder, df in dataframes.items():
    try:
        start = time.time()
        path = f"{silver_path}/{folder}/"

        logger.info("-" * 60)
        logger.info(f"Writing dataset: {folder}")
        logger.info(f"Destination: {path}")

        row_count = df.count()

        (
            df.write
              .format("parquet")
              .mode("overwrite")
              .save(path)
        )

        duration = builtins.round(time.time() - start, 2)

        logger.info(
            f"Successfully wrote '{folder}' "
            f"| Rows: {row_count:,} "
            f"| Duration: {duration} sec"
        )

    except Exception:
        logger.exception(
            f"Failed to write dataset '{folder}' to Silver."
        )

logger.info("=" * 60)
logger.info("Silver Layer Load Completed.")

2026-06-21 01:14:30,897 | INFO | ============================================================
2026-06-21 01:14:30,898 | INFO | Starting Silver Layer Load...
2026-06-21 01:14:30,900 | INFO | ------------------------------------------------------------
2026-06-21 01:14:30,900 | INFO | Writing dataset: ingredients
2026-06-21 01:14:30,901 | INFO | Destination: abfss://silver@chdatalake.dfs.core.windows.net/ingredients/
2026-06-21 01:14:34,897 | INFO | Successfully wrote 'ingredients' | Rows: 18 | Duration: 4.0 sec
2026-06-21 01:14:34,901 | INFO | ------------------------------------------------------------
2026-06-21 01:14:34,903 | INFO | Writing dataset: inventory
2026-06-21 01:14:34,904 | INFO | Destination: abfss://silver@chdatalake.dfs.core.windows.net/inventory/
2026-06-21 01:14:36,595 | INFO | Successfully wrote 'inventory' | Rows: 18 | Duration: 1.69 sec
2026-06-21 01:14:36,598 | INFO | ------------------------------------------------------------
2026-06-21 01:14:36,599 | INFO | Wri